# 02 - Analyze

Copyright (c) Microsoft Corporation. Licensed under the MIT License.

Reads the raw JSON from the Collect stage (same `RUN_ID` folder in the Lakehouse), evaluates configured analyzers against the checklist, and merges to `findings.json`. No Azure/Fabric auth is needed - analyzers are offline. Notebook analysis preserves authoritative workspace/notebook IDs and groups identified hits by IDs; it never reconstructs missing provenance from names for the optional owner projection. Mixed central findings are not copied into the owner report.

## Parameters

In [ ]:
# Parameters (injected by the pipeline)
GITHUB_REPO_URL = "https://github.com/microsoft/fabric-architecture-review.git"
GITHUB_BRANCH = "main"
GITHUB_REF = ""  # pinned release ref from the pipeline (blank = GITHUB_BRANCH tip)
CLIENT_NAME = "Contoso"
ENGAGEMENT_NAME = "Fabric Architecture Review"
REVIEWER_NAME = ""
WORKSPACE_IDS = ""  # same scope as 01 Collect; evidence is bound to it
RUN_ID = "latest"

# Run configuration bindings populated by setup.
FAR_THRESHOLDS_LIBRARY = "FAR_Thresholds"
FAR_CONNECTIONS_LIBRARY = "FAR_Connections"


## Clone framework + install dependencies

In [ ]:
import os, sys, shutil, subprocess
WORK_ROOT = "/tmp/far-run"
REPO_DIR = os.path.join(WORK_ROOT, "repo")
os.makedirs(WORK_ROOT, exist_ok=True)
_url = GITHUB_REPO_URL
_ref = (GITHUB_REF or "").strip() or GITHUB_BRANCH
def _git(args, **kw):
    r = subprocess.run(["git"] + args, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        out = ((r.stderr or "") + (r.stdout or "")).strip()
        raise RuntimeError("git " + " ".join(args) + " failed (rc=" + str(r.returncode) + "): " + out)
    return r

subprocess.run(["git", "config", "--global", "--add", "safe.directory", REPO_DIR])

def _fresh_clone():
    if os.path.isdir(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    _git(["clone", "--branch", _ref, "--depth", "1", _url, REPO_DIR])

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    try:
        _git(["-C", REPO_DIR, "remote", "set-url", "origin", _url])
        _git(["-C", REPO_DIR, "fetch", "--depth", "1", "origin", _ref])
        _git(["-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"])
    except RuntimeError:
        _fresh_clone()
else:
    _fresh_clone()
del _url
sys.path.insert(0, REPO_DIR)
import importlib
# Warm sessions must not reuse framework modules from the previous checkout.
for _module_name in list(sys.modules):
    if _module_name.split(".", 1)[0] in ("collectors", "analyzers", "reports", "orchestration"):
        sys.modules.pop(_module_name, None)
importlib.invalidate_caches()
from pathlib import Path
from orchestration.project import validate_checkout
validate_checkout(Path(REPO_DIR))
if os.path.exists(os.path.join(REPO_DIR, ".git")):
    print("Framework revision:", _git(["-C", REPO_DIR, "rev-parse", "HEAD"]).stdout.strip())
else:
    print("Framework revision: unavailable (source snapshot has no Git metadata)")
os.chdir(REPO_DIR)
_pip = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")])
if _pip.returncode != 0:
    print("WARNING: pip install exited", _pip.returncode, "- a benign dependency-resolver warning is expected on Fabric; continuing.")
print("Repo + deps ready at", REPO_DIR)

## Analyze + merge findings

In [ ]:
import json, os, subprocess, sys
from orchestration.notebook_parameters import optional_string
os.environ["CLIENT_NAME"] = optional_string("CLIENT_NAME", CLIENT_NAME)
os.environ["ENGAGEMENT_NAME"] = optional_string("ENGAGEMENT_NAME", ENGAGEMENT_NAME)
os.environ["REVIEWER_NAME"] = optional_string("REVIEWER_NAME", REVIEWER_NAME)
os.environ["WORKSPACE_IDS"] = optional_string("WORKSPACE_IDS", WORKSPACE_IDS)

LH = "/lakehouse/default/Files"
_BASE = os.path.join(LH, "far")
if RUN_ID == "latest" and os.path.isdir(_BASE):
    _runs = [d for d in os.listdir(_BASE) if os.path.isdir(os.path.join(_BASE, d, "raw"))]
    if _runs:
        RUN_ID = max(_runs, key=lambda d: os.path.getmtime(os.path.join(_BASE, d)))
        print("Resolved RUN_ID=latest ->", RUN_ID)
os.environ["RUN_ID"] = RUN_ID
OUT_DIR = os.path.join(LH, "far", RUN_ID)
RAW_DIR = os.path.join(OUT_DIR, "raw")
from pathlib import Path
from orchestration.review_config import activate
activate(Path(OUT_DIR) / "review_config.json")
import notebookutils
from orchestration.review_workspace import export_review_workspace
export_review_workspace(notebookutils)
if not os.path.isdir(RAW_DIR):
    raise RuntimeError("Raw folder not found: " + RAW_DIR + " -- run the Collect stage first.")
os.environ["OUTPUT_DIR"] = OUT_DIR

# Never let a failed run merge findings or a completion manifest left by an earlier attempt.
for _name in os.listdir(OUT_DIR):
    if _name in ("findings.json", "run_manifest.json") or (_name.startswith("findings_") and _name.endswith(".json")):
        os.remove(os.path.join(OUT_DIR, _name))

_failures = []
def _run(mod, args):
    print("\n==>", mod)
    r = subprocess.run([sys.executable, "-m", mod, *args], cwd=REPO_DIR)
    if r.returncode:
        _failures.append({"module": mod, "returncode": r.returncode})
        print("  ", mod, "exit", r.returncode)

_outputs = []
for mod, fn in [
    ("analyzers.far_review", "findings_architecture.json"),
]:
    _outputs.append(fn)
    _run(mod, ["--raw-dir", RAW_DIR, "--out-dir", OUT_DIR])

_invalid = []
for _name in _outputs:
    _path = os.path.join(OUT_DIR, _name)
    try:
        with open(_path, "r", encoding="utf-8-sig") as _handle:
            _payload = json.load(_handle)
        if not isinstance(_payload, list) or not _payload:
            _invalid.append(_name + " is empty or not a finding list")
    except Exception as _error:
        _invalid.append(_name + ": " + str(_error))
if _failures or _invalid:
    raise RuntimeError("Analysis incomplete: failures=" + json.dumps(_failures) + "; invalid_outputs=" + json.dumps(_invalid))

_run("analyzers.merge_findings", ["--out-dir", OUT_DIR, "--run-id", RUN_ID])
if _failures:
    raise RuntimeError("Finding merge failed: " + json.dumps(_failures))

with open(os.path.join(OUT_DIR, "findings.json"), "r", encoding="utf-8-sig") as _handle:
    _merged = json.load(_handle)
_expected_count = sum(len(json.load(open(os.path.join(OUT_DIR, _name), "r", encoding="utf-8-sig"))) for _name in _outputs)
if not isinstance(_merged, list) or len(_merged) != _expected_count:
    raise RuntimeError("Merged findings completeness check failed")
with open(os.path.join(OUT_DIR, "run_manifest.json"), "r", encoding="utf-8-sig") as _handle:
    _manifest = json.load(_handle)
if _manifest.get("run_id") != RUN_ID or _manifest.get("finding_count") != len(_merged) or _manifest.get("status") != "completed":
    raise RuntimeError("Run manifest completeness check failed")

import notebookutils
notebookutils.notebook.exit("analyzed:" + OUT_DIR + " | findings=" + str(len(_merged)))